In [1]:
# SPDX-License-Identifier: Apache-2.0
"""A reviewer's walkthrough of the certificate, from band edges to held-out validation.

This traces the study's headline deliverable end to end against the committed tables, and it
recomputes rather than reprints: every figure it shows is derived here from
``results/tables/*.csv`` and asserted equal to what the submission claims. If a table changes
and a document does not, this fails.

It is a **repository artefact, not a submission artefact.** The portal accepts no ``.ipynb``
and all five slots are full, so this exists for a reviewer reading the repository -- which is
public, and linked from the proposal's title block.

**It is paired with a committed, executed notebook.** This file is the source of record: it is
linted, tested and diffed like the rest of the code, and ``make walkthrough`` runs it in about a
second. ``make notebook`` executes it into ``walkthrough.ipynb``, which GitHub renders inline so
a reviewer sees every assertion and value without cloning anything. The two are held together by
``test_the_executed_notebook_matches_its_paired_script``, so editing this file without rebuilding
the notebook fails the suite rather than shipping a stale copy.

Deliberately cheap: it reads tables and does arithmetic. Nothing is refitted, so it takes
about a second and needs no GPU. What it verifies is that the *reported* chain is internally
consistent, not that the models retrain -- ``make reproduce`` is what does that.

    .venv/bin/python notebooks/walkthrough.py
"""

"A reviewer's walkthrough of the certificate, from band edges to held-out validation.\n\nThis traces the study's headline deliverable end to end against the committed tables, and it\nrecomputes rather than reprints: every figure it shows is derived here from\n``results/tables/*.csv`` and asserted equal to what the submission claims. If a table changes\nand a document does not, this fails.\n\nIt is a **repository artefact, not a submission artefact.** The portal accepts no ``.ipynb``\nand all five slots are full, so this exists for a reviewer reading the repository -- which is\npublic, and linked from the proposal's title block.\n\n**It is paired with a committed, executed notebook.** This file is the source of record: it is\nlinted, tested and diffed like the rest of the code, and ``make walkthrough`` runs it in about a\nsecond. ``make notebook`` executes it into ``walkthrough.ipynb``, which GitHub renders inline so\na reviewer sees every assertion and value without cloning anything. T

In [2]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

import pandas as pd

# Colab bootstrap, and a no-op everywhere else.  The guard is the import itself: `google.colab`
# exists only inside a Colab runtime, so locally this block is skipped entirely and the cell
# contributes no output to the committed notebook.
#
# The clone is all the setup there is.  This file imports only pandas, which Colab preinstalls,
# and reads seven committed CSVs -- no dataset, no parquet, no run artefacts, no GPU.  So there
# is nothing to `pip install` and nothing to stage; `make reproduce` is what needs an
# environment, and this deliberately is not that.
REPOSITORY = "https://github.com/thedaemon-wizard/Global_Quantum_AI_Challenge_HSBC.git"

try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    checkout = Path(REPOSITORY).stem
    if not Path(checkout).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, checkout], check=True)
    os.chdir(checkout)
    print(f"Colab detected; running against a fresh clone in {Path.cwd()}")


def _repository_root() -> Path:
    """Locate the repository from either a script run or a notebook kernel.

    ``__file__`` is defined when this file runs as a script and **absent when the paired
    notebook runs it in a Jupyter kernel**, so neither `__file__` nor `cwd` alone works in both.
    Walking up for the file that defines the repository is one strategy that works in both and
    raises if the marker is missing, rather than a chain of guesses that silently picks a wrong
    directory and then reads no tables.
    """
    start = Path(globals()["__file__"]).resolve() if "__file__" in globals() else Path.cwd()
    for candidate in (start, *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        f"no pyproject.toml at or above {start}, so the repository root cannot be located; "
        f"run this from inside the repository"
    )


REPO = _repository_root()
TABLES = REPO / "results" / "tables"

pd.set_option("display.width", 100)
pd.set_option("display.max_columns", 20)

## 1. The four blocks

Contiguous, snapped to day boundaries, and disjoint. `D_band` is the only block permitted to
set the band edges; `D_cal` certifies; `D_test` is read once.

In [3]:
splits = pd.read_csv(TABLES / "splits.csv")
temporal = splits[splits["arm"] == "temporal"].set_index("block")
print(temporal[["n_rows", "n_fraud", "n_legit", "day_first", "day_last"]])

total = int(temporal["n_rows"].sum())
print(f"\ntotal rows: {total:,}")
assert total == 590_540, "the four blocks must partition the dataset exactly"

# Blocks must not overlap in time, which is the whole point of the temporal arm.
ordered = temporal.loc[["train", "band", "cal", "test"]]
assert (ordered["day_first"].to_numpy()[1:] > ordered["day_last"].to_numpy()[:-1]).all(), (
    "blocks overlap in time; the temporal split is not a forward holdout"
)
print("blocks are contiguous and strictly ordered in time")

       n_rows  n_fraud  n_legit  day_first  day_last
block                                               
train  356216    12039   344177          0       100
band    58326     2561    55765        101       119
cal     60464     2121    58343        120       140
test   115534     3942   111592        141       181

total rows: 590,540
blocks are contiguous and strictly ordered in time


## 2. Which configurations certify

48 pre-registered `(band budget, alpha, alpha_FN)` cells. A cell certifies when Learn-then-Test
finds an admissible threshold under Holm correction across the grid.

In [4]:
risk = pd.read_csv(TABLES / "riskcontrol.csv")
certified = risk[risk["certified"].astype(bool)]
print(f"{len(certified)} of {len(risk)} configurations certify\n")
print(
    certified[
        ["budget", "alpha", "alpha_fn", "band_lo", "band_hi", "n_legit_band_cal", "selected_lambda"]
    ].to_string(index=False)
)

5 of 48 configurations certify

 budget  alpha  alpha_fn  band_lo  band_hi  n_legit_band_cal  selected_lambda
  0.035   0.25      0.45 0.031981 0.071798              1572         0.058202
  0.050   0.25      0.45 0.025988 0.071798              2259         0.054104
  0.100   0.10      0.45 0.015407 0.071798              4831         0.053650
  0.100   0.15      0.45 0.015407 0.071798              4831         0.053650
  0.100   0.25      0.45 0.015407 0.071798              4831         0.042632


**The certificate must not be vacuous.** A selected threshold sitting at the upper band edge
flags nothing, so its risk is structurally zero and the guarantee says nothing. An earlier
version of this table was exactly that: 32 cells certified, every one at the edge.

In [5]:
margin = (certified["band_hi"] - certified["selected_lambda"]).min()
print(f"smallest margin below the upper band edge: {margin:.4f}")
assert margin > 0, "a threshold at the band edge certifies an empty flagged set"

smallest margin below the upper band edge: 0.0136


**What limits the reach.** Nothing certifies at the tightest band budget. The mechanism is
sample size acting through the concentration bound, not the class-conditional degeneracy
floor -- the floor is `(1/alpha) - 1`, which is more than forty times below the rows
available and cannot bind anywhere on this grid.

In [6]:
tightest = risk[risk["budget"] == risk["budget"].min()]
rows_available = int(tightest["n_legit_band_cal"].min())
floor_at_five_pct = (1.0 / 0.05) - 1.0
print(f"tightest band budget: {tightest['budget'].iloc[0]}")
print(f"  certifies:            {int(tightest['certified'].astype(bool).sum())} of {len(tightest)}")
print(f"  in-band legit rows:   {rows_available}")
print(f"  degeneracy floor:     {floor_at_five_pct:.0f}")
assert floor_at_five_pct < rows_available, (
    "if the floor exceeded the rows available it would be the binding constraint"
)
print("  -> the floor cannot bind; the concentration bound is what does")

tightest band budget: 0.02
  certifies:            0 of 12
  in-band legit rows:   847
  degeneracy floor:     19
  -> the floor cannot bind; the concentration bound is what does


## 3. Does the certificate hold on data it never saw?

This is H5. The certified cells are applied to `D_test` unchanged -- same band edges, same
lambda, no recalibration -- and the realised band-conditional false-decline rate is compared
with the alpha each was certified at.

In [7]:
h5 = pd.read_csv(TABLES / "h5_validation.csv")
view = h5[["band_budget", "alpha", "n_legit_band_test", "declined", "realised_risk"]].copy()
view["risk / alpha"] = (h5["realised_risk"] / h5["alpha"]).round(3)
view["holds"] = h5["holds_at_alpha"]
print(view.to_string(index=False))

# The realised risk is recomputed from the counts rather than read, so a wrong risk column
# would surface here instead of propagating.
recomputed = h5["declined"] / h5["n_legit_band_test"]
assert (recomputed - h5["realised_risk"]).abs().max() < 1e-9, (
    "realised_risk does not equal declined / n_legit_band_test"
)

worst = (h5["realised_risk"] / h5["alpha"]).max()
# Assert before announcing. Printed first, a failing table produced the line "all 5 hold" and
# only then died, so the transcript of a failed run stated the opposite of its own verdict.
assert bool(h5["holds_at_alpha"].all()), "a certified configuration failed on held-out data"
print(f"\nall {len(h5)} hold; the tightest uses {worst:.3f} of its budget")

 band_budget  alpha  n_legit_band_test  declined  realised_risk  risk / alpha  holds
       0.035   0.25               3314       580       0.175015         0.700   True
       0.050   0.25               4835       830       0.171665         0.687   True
       0.100   0.10              10021       858       0.085620         0.856   True
       0.100   0.15              10021       858       0.085620         0.571   True
       0.100   0.25              10021      1762       0.175831         0.703   True

all 5 hold; the tightest uses 0.856 of its budget


**Read it for what it is.** Holding a ceiling of 0.10 to 0.25 is a real check that the
machinery transfers, and a weak one, because those ceilings are loose. Amendment A1 predicted
that from the band's sample size before the arm ran.

## 4. Where the guarantee breaks: time

Split-conformal coverage is judged against the exact Beta-Binomial law for the sample size,
not against alpha. Checking `empirical <= alpha` is a one-sided test against the wrong null:
on a correctly calibrated system it passes about half the time.

In [8]:
seeds = pd.read_csv(TABLES / "coverage_by_arm_seeds.csv")
summary = (
    seeds.groupby(["arm", "alpha"])
    .agg(mean_ratio=("ratio", "mean"), outside=("inside", lambda c: int((~c.astype(bool)).sum())))
    .reset_index()
)
print(summary.to_string(index=False))

          arm  alpha  mean_ratio  outside
card_disjoint  0.001    0.881986        1
card_disjoint  0.002    0.904024        0
card_disjoint  0.005    0.917043        0
card_disjoint  0.010    0.916240        2
   stratified  0.001    0.961614        0
   stratified  0.002    1.018644        0
   stratified  0.005    1.039175        0
   stratified  0.010    1.008116        0
     temporal  0.001    0.942720        0
     temporal  0.002    1.313714        4
     temporal  0.005    1.455660        5
     temporal  0.010    1.436304        5


The temporal arm falls outside on every seed at the two loosest levels; the stratified arm on
none, at any level. The card-disjoint arm is **not clean, and the two levels fail
differently** -- which is the distinction an earlier draft of the proposal got wrong.

In [9]:
card = seeds[(seeds["arm"] == "card_disjoint") & (~seeds["inside"].astype(bool))]
print(card[["seed", "alpha", "ratio", "conservative"]].to_string(index=False))

over = card[card["conservative"].astype(bool)]
under = card[~card["conservative"].astype(bool)]
print(f"\nover-covering (conservative, guarantee intact): {len(over)}")
print(f"under-covering (the direction that breaches):   {len(under)}")
assert len(under) == 1 and (under["ratio"] > 1).all(), (
    "the anti-conservative cell is what stops this arm being reported as a pass"
)
print("-> one seed at the tightest level runs the other way, so the arm is not a pass")

    seed  alpha    ratio  conservative
20260831  0.010 0.785315          True
20260831  0.001 1.524539         False
20260901  0.010 0.794822          True

over-covering (conservative, guarantee intact): 2
under-covering (the direction that breaches):   1
-> one seed at the tightest level runs the other way, so the arm is not a pass


## 5. And the size of the temporal breach depends on where you calibrate

Re-running at five rolling calibration origins. The deviation is not a fixed factor, and the
earliest origin reverses direction entirely -- which is why the study reports no single
inflation number.

In [10]:
rolling = pd.read_csv(TABLES / "rolling_origin.csv")
print(rolling[["cal_start", "ratio", "verdict", "band_low", "band_high"]].to_string(index=False))
print(f"\nratio ranges {rolling['ratio'].min():.3f} to {rolling['ratio'].max():.3f}")
assert rolling["ratio"].min() < 1.0 < rolling["ratio"].max(), (
    "origins must straddle 1.0 for the direction-reversal claim to hold"
)

 cal_start    ratio      verdict  band_low  band_high
       101 0.683536 conservative       472        641
       111 0.892909       inside       437        595
       121 1.410932     BREACHED       468        638
       131 1.198368     BREACHED       494        674
       141 1.126431       inside       457        625

ratio ranges 0.684 to 1.411


## 6. The quantum arms, for completeness

Both are negative and both are reported in the body. The kernel never reached the task: the
a-priori screens rejected every configuration before it was run.

In [11]:
screens = pd.read_csv(TABLES / "screens.csv")
both = screens["passes_conditioning"].astype(bool) & screens["passes_distinctness"].astype(bool)
passed_both = int(both.sum())
print(f"{len(screens)} configurations screened")
print(f"  pass conditioning:  {int(screens['passes_conditioning'].astype(bool).sum())}")
print(f"  pass both:          {passed_both}")
print(f"  max RBF correlation: {screens['rbf_correlation'].max():.4f}")
assert passed_both == 0, "the kernel arm was stopped by its own pre-registered screens"

120 configurations screened
  pass conditioning:  28
  pass both:          0
  max RBF correlation: 1.0000


The tensor network ran. Every interval contains zero, and the comparison is underpowered
against its own pre-registered ceiling -- so what survives is a non-superiority bound, not a
null result and not evidence that the tensor network is worse.

In [12]:
mps = pd.read_csv(TABLES / "mps_h4.csv")
print(mps[["bond_dimension", "ap_difference", "ci_low", "ci_high"]].to_string(index=False))
assert ((mps["ci_low"] < 0) & (mps["ci_high"] > 0)).all(), "an interval excludes zero"
print(f"\nevery interval contains zero; upper bound on any gain: +{mps['ci_high'].max():.4f} AP")

 bond_dimension  ap_difference    ci_low  ci_high
              4      -0.020530 -0.059302 0.014900
              8      -0.023447 -0.063001 0.012232
             16      -0.022452 -0.061253 0.013594
             32      -0.017198 -0.058198 0.019998

every interval contains zero; upper bound on any gain: +0.0200 AP


In [13]:
def main() -> int:
    """Closing report, not the body of the walkthrough.

    Every cell above runs at import, which the ``# %%`` paired-cell format requires, so the
    assertions have already fired by the time this is called and it cannot report a failure
    itself.  It exists so ``make walkthrough`` has a named exit path rather than depending on
    module import for its status.
    """
    print("\nWalkthrough complete: every assertion above holds against the committed tables.")
    return 0


if __name__ == "__main__":
    status = main()
    # A Jupyter kernel also sets ``__name__ == "__main__"``, so the guard alone does not
    # distinguish `make walkthrough` from the paired notebook -- and raising SystemExit inside a
    # kernel aborts the cell, which nbconvert reports as a failed notebook even at status 0.
    # ``__file__`` is what actually separates the two: it exists when there is a script to exit
    # from.  So the status is raised for the Makefile and simply returned for the notebook.
    if "__file__" in globals():
        raise SystemExit(status)


Walkthrough complete: every assertion above holds against the committed tables.
